# Week 5 — Your First CNN (STARTER)
### The Computer Vision Internship | VisionAI Lagos

Tunde: 'Tell me what a convolutional layer does — without using the word filter.'

## Part 1 — What Convolution Does

> ⏸️ **Pause and Predict**
> The CNN has 4 conv blocks each with MaxPool2d(2). Starting from 64x64:
> what is the spatial resolution after each block? Write: 64 → ? → ? → ? → ?

In [ ]:
# Manual convolution with hand-crafted filters
import torch.nn as nn

row=df.iloc[0]; img_arr=np.array(Image.open(f"datasets/images/{row['filename']}"))
img_t=torch.from_numpy(img_arr.transpose(2,0,1)).float().unsqueeze(0)/255.0
print(f"Input: {img_t.shape}")  # (1, 3, 64, 64)

filters={"Horizontal edge":torch.tensor([[[-1,-1,-1],[0,0,0],[1,1,1]]]).float().unsqueeze(0).repeat(1,3,1,1),
          "Vertical edge":torch.tensor([[[-1,0,1],[-1,0,1],[-1,0,1]]]).float().unsqueeze(0).repeat(1,3,1,1),
          "Blur":torch.ones(1,3,3,3).float()/9}

fig,axes=plt.subplots(1,4,figsize=(16,4))
axes[0].imshow(img_arr); axes[0].set_title("Original"); axes[0].axis("off")
for i,(name,kernel) in enumerate(filters.items(),1):
    conv=nn.Conv2d(3,1,kernel_size=3,padding=1,bias=False)
    conv.weight=nn.Parameter(kernel)
    with torch.no_grad(): feat=conv(img_t).squeeze().numpy()
    axes[i].imshow(feat,cmap="RdBu_r"); axes[i].set_title(f"{name}"); axes[i].axis("off")
plt.suptitle("Hand-Crafted Filters vs Learnable Filters",fontweight="bold")
plt.tight_layout(); plt.savefig("outputs/convolution_demo.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()
print("A CNN learns optimal filter weights automatically from the labels.")

## Part 2 — Build the CNN

In [ ]:
class SatelliteCNN(nn.Module):
    def __init__(self,num_classes=4):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(inplace=True),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(inplace=True),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(inplace=True),nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(inplace=True),nn.MaxPool2d(2),
        )
        self.gap=nn.AdaptiveAvgPool2d(1)
        self.classifier=nn.Sequential(nn.Dropout(0.4),nn.Linear(256,128),nn.ReLU(inplace=True),
                                       nn.Dropout(0.3),nn.Linear(128,num_classes))
    def forward(self,x): return self.classifier(self.gap(self.features(x)).flatten(1))
model=SatelliteCNN(num_classes=4).to(DEVICE)
total=sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")
dummy=torch.randn(4,3,64,64).to(DEVICE)
out=model(dummy)
print(f"Forward pass: {dummy.shape} -> {out.shape}")
print("\nSpatial resolution at each block:")
x=dummy
for i,layer in enumerate(model.features):
    x=layer(x)
    if isinstance(layer,nn.MaxPool2d):
        print(f"  After block {i//4+1} MaxPool: {x.shape[2]}x{x.shape[3]}")

## Part 3 — Training

> 🧠 **What Will This Output?**
> Random init 4-class loss = ln(4) ≈ 1.386. If your first epoch loss is much higher than 1.4, something is wrong. What could cause this?

In [ ]:
model=SatelliteCNN(num_classes=4).to(DEVICE)
criterion=nn.CrossEntropyLoss(); opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,patience=3,factor=0.5)
N_EPOCHS=20; history={"train_loss":[],"val_loss":[],"train_f1":[],"val_f1":[]}
best_val_f1,best_epoch=0,0
print(f"Training SatelliteCNN ({sum(p.numel() for p in model.parameters()):,} params)...")
print(f"{'Ep':>4} {'TrL':>8} {'VaL':>8} {'TrF1':>7} {'VaF1':>7} {'LR':>9}")
print("-"*48)
for ep in range(1,N_EPOCHS+1):
    t0=time.time()
    tr_l,tr_f=train_epoch(model,train_loader,criterion,opt,DEVICE)
    va_l,va_f=eval_epoch(model,val_loader,criterion,DEVICE)
    scheduler.step(va_l); lr=opt.param_groups[0]["lr"]
    for k,v in zip(["train_loss","val_loss","train_f1","val_f1"],[tr_l,va_l,tr_f,va_f]):
        history[k].append(v)
    if va_f>best_val_f1:
        best_val_f1=va_f; best_epoch=ep
        torch.save(model.state_dict(),"models/cnn_scratch_best.pth")
    print(f"{ep:>4} {tr_l:>8.4f} {va_l:>8.4f} {tr_f:>7.3f} {va_f:>7.3f} {lr:>9.6f}  ({time.time()-t0:.1f}s)")
print(f"Best val F1: {best_val_f1:.3f} at epoch {best_epoch}")

## Part 4 — Training Curves

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].plot(history["train_loss"],label="Train",color="#2E75B6")
axes[0].plot(history["val_loss"],label="Val",color="#C0392B",marker="o",markersize=3)
axes[0].set_title("Cross-Entropy Loss",fontweight="bold",loc="left")
axes[0].legend(); axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)
axes[1].plot(history["train_f1"],label="Train",color="#2E75B6")
axes[1].plot(history["val_f1"],label="Val",color="#C0392B",marker="o",markersize=3)
axes[1].axhline(max(history["val_f1"]),color="gray",linewidth=0.8,linestyle="--")
axes[1].set_title(f"Weighted F1 (best val: {max(history['val_f1']):.3f})",fontweight="bold",loc="left")
axes[1].legend(); axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)
plt.suptitle("SatelliteCNN Training — VisionAI Lagos",fontweight="bold")
plt.tight_layout(); plt.savefig("outputs/cnn_curves.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()
gap=max(history["train_f1"])-max(history["val_f1"])
print(f"Train-val gap: {gap:.3f} ({'overfitting' if gap>0.1 else 'acceptable'})")

## Part 5 — Filter Visualisation and Feature Maps

> 🧠 **What Will This Output?**
> After 20 training epochs, will the first-layer filters look like random noise, structured patterns, or something in between? What would each outcome tell you about training?

In [ ]:
# Load best model and visualise first-layer filters
model.load_state_dict(torch.load("models/cnn_scratch_best.pth",map_location=DEVICE))
model.eval()
first_layer_filters=model.features[0].weight.data.cpu()
print(f"First layer filters: {first_layer_filters.shape}")
fig,axes=plt.subplots(4,8,figsize=(18,10))
for i,ax in enumerate(axes.flatten()):
    if i<first_layer_filters.shape[0]:
        f=first_layer_filters[i].permute(1,2,0).numpy()
        f=(f-f.min())/(f.max()-f.min()+1e-8)
        ax.imshow(np.clip(f,0,1)); ax.set_title(f"F{i+1}",fontsize=7)
    ax.axis("off")
plt.suptitle("Learned First-Layer Filters (edge, colour, texture detectors)",fontweight="bold")
plt.tight_layout(); plt.savefig("outputs/learned_filters.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

In [ ]:
# Feature map progression — what the model sees at each depth
row=df[df["land_use"]=="green_space"].iloc[0]
img_pil=Image.open(f"datasets/images/{row['filename']}").convert("RGB")
img_t=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])(img_pil).unsqueeze(0).to(DEVICE)

fmaps={}
def hook(name): return lambda m,i,o: fmaps.update({name:o.detach().cpu()})
hooks=[layer.register_forward_hook(hook(f"block_{j}")) for j,layer in enumerate(model.features)
       if isinstance(layer,nn.MaxPool2d)]
with torch.no_grad(): _=model(img_t)
for h in hooks: h.remove()

fig,axes=plt.subplots(len(fmaps),9,figsize=(20,4*len(fmaps)))
for ri,(bname,fmap) in enumerate(sorted(fmaps.items())):
    axes[ri][0].imshow(np.array(img_pil)); axes[ri][0].axis("off")
    axes[ri][0].set_ylabel(f"{bname}\n{fmap.shape[2]}x{fmap.shape[3]}",rotation=0,labelpad=55,fontsize=8)
    for ci in range(1,9):
        if ci-1<fmap.shape[1]:
            axes[ri][ci].imshow(fmap[0,ci-1].numpy(),cmap="viridis")
        axes[ri][ci].axis("off")
plt.suptitle("Feature Map Progression (Block 1=interpretable -> Block 4=abstract)",fontweight="bold")
plt.tight_layout(); plt.savefig("outputs/feature_maps.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()
print("Block 4 features are 4x4 — the black box transition. GradCAM (Week 11) looks inside.")

## Part 6 — Test Evaluation

In [ ]:
model.load_state_dict(torch.load("models/cnn_scratch_best.pth",map_location=DEVICE))
model.eval()
preds_all,labels_all,cities_all=[],[],[]
with torch.no_grad():
    for imgs,lbls,cities in test_loader:
        preds_all.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy())
        labels_all.extend(lbls.numpy()); cities_all.extend(cities)
test_f1=f1_score(labels_all,preds_all,average="weighted",zero_division=0)
print(f"Test weighted F1: {test_f1:.3f}")
print(classification_report(labels_all,preds_all,target_names=CLASSES,zero_division=0))

cm=confusion_matrix(labels_all,preds_all)
fig,ax=plt.subplots(figsize=(7,6))
sns.heatmap(pd.DataFrame(cm,index=CLASSES,columns=CLASSES),annot=True,fmt="d",cmap="Blues",ax=ax)
ax.set_title(f"Confusion Matrix — CNN scratch (F1={test_f1:.3f})",fontweight="bold",loc="left")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.savefig("outputs/cnn_confusion.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

# Recall: Book 2 label leakage was fitting vectoriser on test data.
# In CNN: the equivalent is fitting normalisation stats on test data.
# We used ImageNet mean/std — no leakage. Always check this.

## Common Mistakes

| Mistake | Fix |
|---------|-----|
| Forgetting model.train()/model.eval() | BatchNorm/Dropout behave incorrectly |
| Not zeroing gradients | Accumulate across batches |
| Augmenting val/test | Already caught in Week 3 |
| Evaluating last epoch not best | Load best checkpoint before test |

## Challenge Task

> Experiment with kernel size. Replace all 3x3 conv filters in Block 1 with 5x5. How does parameter count, training speed, and final F1 change? What does a larger kernel capture for satellite imagery?

## Week 5 Complete

Next: `week6_improving_cnn_STARTER.ipynb`

*The Computer Vision Internship · VisionAI Lagos*

## ✅ By completing Week 5, you can now:

- Build and train a CNN from scratch on the satellite dataset
- Explain what each layer does in one plain-English sentence
- Plot training and validation loss and diagnose the failure mode
- Visualise learned filters in the first convolutional layer

📤 **GitHub:** Push week5_cnn_from_scratch_STARTER.ipynb. Commit: "Week 5: CNN trained, filters visualised"


---

## 📚 Get the Full Book

This notebook is part of **The Computer Vision Internship** — Book 3 of the InternshipInABook™ Series.

👉 **[Get the book on Selar → [SELAR_LINK_PLACEHOLDER]]**

---
*InternshipInABook™ · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook/cv-internship-in-a-book)*
